In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [3]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *
from kret_sandbox.VIS import dtt

In [4]:
config = EnvConfig()

In [5]:
"""
Define the observation space for the environment.
    - globals: day of week (sin, cos), time of day (sin, cos), weather_one_hot(3) (7,) #TODO stretch goal: add demand per node, next step speed limits
    - supply_demand_ratio: current supply-demand ratio, vehicle-per-node, lambda-per-node (3,)
    - vehicles: (num_vehicles, 7) -> [loc_x_norm, loc_y_norm, battery, status_one_hot(4)]
    - pending_requests: (max_pending_requests, 9) -> [pickup_x_norm, pickup_y_norm, dropoff_x_norm, dropoff_y_norm, distance_meters, est_cost, max_wait_time, wait_time, status_one_hot(1)] # TODO add max wait time before
    - active_rides: (num_vehicles, 8) -> [pickup_x_norm, pickup_y_norm, dropoff_x_norm, dropoff_y_norm, price, total_trip_distance, trip_distance_remaining, pickup_distance_remaining]
    - dispatch_mask: (num_vehicles) -> 1 if vehicle can be dispatched to request, else 0
    - pricing_mask: (max_pending_requests) -> 1 if request needs pricing decision, else 0
"""

'\nDefine the observation space for the environment.\n    - globals: day of week (sin, cos), time of day (sin, cos), weather_one_hot(3) (7,) #TODO stretch goal: add demand per node, next step speed limits\n    - supply_demand_ratio: current supply-demand ratio, vehicle-per-node, lambda-per-node (3,)\n    - vehicles: (num_vehicles, 7) -> [loc_x_norm, loc_y_norm, battery, status_one_hot(4)]\n    - pending_requests: (max_pending_requests, 9) -> [pickup_x_norm, pickup_y_norm, dropoff_x_norm, dropoff_y_norm, distance_meters, est_cost, max_wait_time, wait_time, status_one_hot(1)] # TODO add max wait time before\n    - active_rides: (num_vehicles, 8) -> [pickup_x_norm, pickup_y_norm, dropoff_x_norm, dropoff_y_norm, price, total_trip_distance, trip_distance_remaining, pickup_distance_remaining]\n    - dispatch_mask: (num_vehicles) -> 1 if vehicle can be dispatched to request, else 0\n    - pricing_mask: (max_pending_requests) -> 1 if request needs pricing decision, else 0\n'

In [6]:
# config = ActiveRideDF.space_config(config, 10)
# config["shape"]

In [28]:
env = RideShareEnv(config=config)
G = env.G

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:424: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


In [51]:
s, info = env.reset()

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:424: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


In [52]:
veh = env.observation_curr["vehicles"]
req = env.observation_curr["pending_requests"]
rides = env.observation_curr["active_rides"]
type(veh), type(req), type(rides)

(waymo_agent.data_classes.vehicles.VehicleDF,
 waymo_agent.data_classes.requests.RequestDF,
 waymo_agent.data_classes.active_rides.ActiveRideDF)

In [53]:
# req

In [54]:
action = env.action_space.sample()

In [64]:
action["reposition"]

array([[ 0.82618177, -0.3133576 ],
       [-0.17417742, -0.5248305 ],
       [ 0.22254726, -0.24447495],
       [-0.49680778, -0.7396577 ],
       [ 0.33238575,  0.306785  ],
       [-0.7616012 , -0.46450344],
       [ 0.03314291, -0.9091629 ],
       [-0.06610635,  0.71686155],
       [ 0.8312764 ,  0.04152099],
       [-0.58359474, -0.8509898 ],
       [ 0.7823808 , -0.14035231],
       [ 0.88225585,  0.7015438 ],
       [ 0.6279004 , -0.22072431],
       [-0.70404255, -0.01374481],
       [ 0.73607796, -0.19429769],
       [ 0.6817838 ,  0.7431241 ],
       [-0.793819  , -0.67539084],
       [-0.7054118 ,  0.63505864],
       [-0.511231  ,  0.5612751 ],
       [-0.4634606 ,  0.1563003 ],
       [ 0.4765586 , -0.43829173],
       [ 0.1000994 ,  0.96703607],
       [-0.04606769,  0.45838344],
       [ 0.5733397 , -0.7138458 ],
       [-0.04812653, -0.3090575 ]], dtype=float32)

In [57]:
obs, reward, term, trunc, info = env.step(action)

In [58]:
RequestDF.default_vals.get("max_wait_time")

15

In [59]:
np.timedelta64(15, "m")

np.timedelta64(15,'m')

In [60]:
filler_requests = RequestDF.generate_empty(num_rows=4)
filler_requests.max_wait_time

0   0 days 00:15:00
1   0 days 00:15:00
2   0 days 00:15:00
3   0 days 00:15:00
Name: max_wait_time, dtype: timedelta64[ns]